# Full-Sample Analysis: Monetary & Fiscal Agreement and Stance

This notebook replicates the data transformations from `temp/13.create_final_dataset.py` and
the statistical analysis / visualizations from `temp/14.analysis_full_sample.ipynb`,
adapted to the new inference pipeline output format.

**Scope**: Monetary and Fiscal sectors only (no macro-economic variables).

In [17]:
import ast
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import data_vis_utils as dv

In [32]:
# ── Configuration ──────────────────────────────────────────────────────────────
OUTPUT_DIR = Path("/data/home/xiong/data/Fund/CSR/Tractions/output")

DOC_LEVEL_PATH            = OUTPUT_DIR / "df_aiv.csv"
AGREEMENT_MONETARY_PATH   = OUTPUT_DIR / "agreement_monetary_results.csv"
AGREEMENT_FISCAL_PATH     = OUTPUT_DIR / "agreement_fiscal_results.csv"
STANCE_MONETARY_PATH      = OUTPUT_DIR / "stance_monetary_results.csv"
STANCE_FISCAL_PATH        = OUTPUT_DIR / "stance_fiscal_results.csv"
FINAL_DATASET_PATH        = OUTPUT_DIR / "df_fin.csv"


START_YEAR = 2015
END_YEAR   = 2023

## Data Loading

In [19]:
# ── Load raw data ─────────────────────────────────────────────────────────────
df_aiv = pd.read_csv(DOC_LEVEL_PATH)
df_aiv = df_aiv[df_aiv["staff_verified"] == True].copy()
df_aiv["year"] = pd.to_numeric(df_aiv["Year from title"], errors="coerce").astype("Int64")
df_aiv = df_aiv[df_aiv["year"].notna()].copy()
df_aiv = df_aiv[(df_aiv["year"] >= START_YEAR) & (df_aiv["year"] <= END_YEAR)].copy()
df_aiv["year"] = df_aiv["year"].astype(int)
df_aiv["income_group"] = df_aiv["Primary Country Code"].apply(
    lambda c: dv.classify_income_group_from_code(c, other_label="LC")
)
print(f"df_aiv: {df_aiv.shape}")

df_agree_mon = pd.read_csv(AGREEMENT_MONETARY_PATH)
df_agree_mon = dv.filter_year_range(df_agree_mon, start_year=START_YEAR, end_year=END_YEAR)
print(f"agreement_monetary: {df_agree_mon.shape}")

df_agree_fis = pd.read_csv(AGREEMENT_FISCAL_PATH)
df_agree_fis = dv.filter_year_range(df_agree_fis, start_year=START_YEAR, end_year=END_YEAR)
print(f"agreement_fiscal: {df_agree_fis.shape}")

df_stance_mon = pd.read_csv(STANCE_MONETARY_PATH)
df_stance_mon = dv.filter_year_range(df_stance_mon, start_year=START_YEAR, end_year=END_YEAR)
print(f"stance_monetary: {df_stance_mon.shape}")

df_stance_fis = pd.read_csv(STANCE_FISCAL_PATH)
df_stance_fis = dv.filter_year_range(df_stance_fis, start_year=START_YEAR, end_year=END_YEAR)
print(f"stance_fiscal: {df_stance_fis.shape}")

df_aiv: (981, 23)
agreement_monetary: (807, 9)
agreement_fiscal: (971, 9)
stance_monetary: (1440, 14)
stance_fiscal: (1945, 13)


## Data Transformation

Adapted from `temp/13.create_final_dataset.py` to new per-sector CSV format.

In [20]:
# ── Monetary stance: pivot long → wide ─────────────────────────────────────────
df_mon_wide = dv.pivot_stance_wide(
    df_stance_mon,
    author_col="type",
    stance_cols=["stance_current", "stance_future"],
)
df_mon_wide = df_mon_wide.rename(columns={
    "staff_stance_current":  "mon_stance_current_staff",
    "buff_stance_current":   "mon_stance_current_buff",
    "staff_stance_future":   "mon_stance_future_staff",
    "buff_stance_future":    "mon_stance_future_buff",
})

# Clean NaN-like strings  # Ref: 13.create_final_dataset.py L58-62
for col in ["mon_stance_current_staff", "mon_stance_current_buff",
            "mon_stance_future_staff", "mon_stance_future_buff"]:
    df_mon_wide[col] = df_mon_wide[col].fillna("irrelevant")
    df_mon_wide[col] = df_mon_wide[col].apply(
        lambda x: "irrelevant" if str(x).strip().lower() in ["nan", ""] else x
    )

# Consolidate current stance categories  # Ref: 13.create_final_dataset.py L65-66
for col in ["mon_stance_current_staff", "mon_stance_current_buff"]:
    df_mon_wide[col] = df_mon_wide[col].apply(
        lambda x: "restrictive" if x in ["moderately tight", "tightening bias"]
        else "neutral" if x in ["close to neutral"]
        else x
    )

print(f"Monetary stance wide: {df_mon_wide.shape}")
df_mon_wide.head()

Monetary stance wide: (807, 8)


,Print ISBN,topic,country,year,mon_stance_current_buff,mon_stance_current_staff,mon_stance_future_buff,mon_stance_future_staff
0,9781475515039,Monetary,Uruguay,2015,restrictive,restrictive,no change,no change
1,9781475517231,Monetary,United Kingdom,2016,unclear,accommodative,tightening bias,no change
2,9781475517408,Monetary,Qatar,2015,irrelevant,accommodative,irrelevant,no change
3,9781475517682,Monetary,Jamaica,2016,unclear,neutral,loosening bias,loosening bias
4,9781475517927,Monetary,Hungary,2016,accommodative,accommodative,loosening bias,loosening bias


In [21]:
# ── Monetary: numeric stance scores & agreement variables ─────────────────────
stance_future_dict = {"tightening": 5, "tightening bias": 4, "no change": 3,  # Ref: 13.create_final_dataset.py L68
                      "loosening bias": 2, "loosening": 1}
stance_current_dict = {"accommodative": 0, "neutral": 1, "restrictive": 2}    # Ref: 13.create_final_dataset.py L75

# Future stance numeric  # Ref: 13.create_final_dataset.py L69-71
df_mon_wide["mon_stance_future_staff_num"] = df_mon_wide["mon_stance_future_staff"].map(stance_future_dict)
df_mon_wide["mon_stance_future_buff_num"]  = df_mon_wide["mon_stance_future_buff"].map(stance_future_dict)
df_mon_wide["mon_agreement_stance_future_num"] = (
    df_mon_wide["mon_stance_future_staff_num"] - df_mon_wide["mon_stance_future_buff_num"]
)  #### follow how it was done in xiaorui's code  ; this is important, it is a differnt type of definiation of agreement 

# Categorize future stance difference  # Ref: 13.create_final_dataset.py L72
df_mon_wide["mon_agreement_stance_future_cate1"] = df_mon_wide["mon_agreement_stance_future_num"].apply(
    lambda x: "major difference" if abs(x) >= 3 else "some difference" if abs(x) == 2
    else "minor difference" if abs(x) == 1 else "no difference" if x == 0 else "irrelevant"
    if pd.isna(x) else "irrelevant"
)
# Ref: 13.create_final_dataset.py L73
df_mon_wide["mon_agreement_stance_future_cate2"] = df_mon_wide["mon_agreement_stance_future_num"].apply(
    lambda x: "significantly tighter" if x >= 3 else "tighter" if x == 2
    else "moderately tighter" if x == 1 else "same" if x == 0
    else "moderately looser" if x == -1 else "looser" if x == -2
    else "significantly looser" if x <= -3 else "irrelevant"
    if pd.isna(x) else "irrelevant"
)

# Current stance numeric  # Ref: 13.create_final_dataset.py L76-78
df_mon_wide["mon_stance_current_staff_num"] = df_mon_wide["mon_stance_current_staff"].map(stance_current_dict)
df_mon_wide["mon_stance_current_buff_num"]  = df_mon_wide["mon_stance_current_buff"].map(stance_current_dict)
df_mon_wide["mon_agreement_stance_current_num"] = (
    df_mon_wide["mon_stance_current_staff_num"] - df_mon_wide["mon_stance_current_buff_num"]
)

# Agreement on current stance (categorical)  # Ref: 13.create_final_dataset.py L81
def _mon_agree_current(row):
    s, b = row["mon_stance_current_staff"], row["mon_stance_current_buff"]
    for v in [s, b]:
        if v in ["", "nan", "n", "unclear", "irrelevant"] or pd.isna(v):
            return "irrelevant"
    return "mostly agree" if s == b else "disagreement exists"

df_mon_wide["mon_agreement_stance_current"] = df_mon_wide.apply(_mon_agree_current, axis=1)

# Agreement on future stance (within 1 step = mostly agree)  # Ref: 13.create_final_dataset.py L83
def _mon_agree_future(row):
    s, b = row["mon_stance_future_staff"], row["mon_stance_future_buff"]
    for v in [s, b]:
        if v in ["", "nan", "n", "unclear", "irrelevant"] or pd.isna(v):
            return "irrelevant"
    diff = row["mon_agreement_stance_future_num"]
    if pd.isna(diff):
        return "irrelevant"
    return "mostly agree" if abs(diff) <= 1 else "disagreement exists"

df_mon_wide["mon_agreement_stance_future"] = df_mon_wide.apply(_mon_agree_future, axis=1)

print("Mon future stance agreement:")
print(df_mon_wide["mon_agreement_stance_future"].value_counts())

Mon future stance agreement:
mon_agreement_stance_future
mostly agree           443
irrelevant             281
disagreement exists     83
Name: count, dtype: int64


In [22]:
df_mon_wide.head()

,Print ISBN,topic,country,year,mon_stance_current_buff,mon_stance_current_staff,mon_stance_future_buff,mon_stance_future_staff,mon_stance_future_staff_num,mon_stance_future_buff_num,mon_agreement_stance_future_num,mon_agreement_stance_future_cate1,mon_agreement_stance_future_cate2,mon_stance_current_staff_num,mon_stance_current_buff_num,mon_agreement_stance_current_num,mon_agreement_stance_current,mon_agreement_stance_future
0,9781475515039,Monetary,Uruguay,2015,restrictive,restrictive,no change,no change,3.0,3.0,0.0,no difference,same,2.0,2.0,0.0,mostly agree,mostly agree
1,9781475517231,Monetary,United Kingdom,2016,unclear,accommodative,tightening bias,no change,3.0,4.0,-1.0,minor difference,moderately looser,0.0,NaN,NaN,irrelevant,mostly agree
2,9781475517408,Monetary,Qatar,2015,irrelevant,accommodative,irrelevant,no change,3.0,NaN,NaN,irrelevant,irrelevant,0.0,NaN,NaN,irrelevant,irrelevant
3,9781475517682,Monetary,Jamaica,2016,unclear,neutral,loosening bias,loosening bias,2.0,2.0,0.0,no difference,same,1.0,NaN,NaN,irrelevant,mostly agree
4,9781475517927,Monetary,Hungary,2016,accommodative,accommodative,loosening bias,loosening bias,2.0,2.0,0.0,no difference,same,0.0,0.0,0.0,mostly agree,mostly agree


In [23]:
# ── Fiscal stance: pivot long → wide ──────────────────────────────────────────
df_fis_wide = dv.pivot_stance_wide(
    df_stance_fis,
    author_col="type",
    stance_cols=["stance_near_term"],
)
df_fis_wide = df_fis_wide.rename(columns={
    "staff_stance_near_term": "fis_stance_near_term_staff",
    "buff_stance_near_term":  "fis_stance_near_term_buff",
})

# Clean NaN-like strings  # Ref: 13.create_final_dataset.py L91-94
for col in ["fis_stance_near_term_staff", "fis_stance_near_term_buff"]:
    df_fis_wide[col] = df_fis_wide[col].fillna("irrelevant")
    df_fis_wide[col] = df_fis_wide[col].apply(
        lambda x: "irrelevant" if str(x).strip().lower() in ["nan", ""] else x
    )

print(f"Fiscal stance wide: {df_fis_wide.shape}")
df_fis_wide.head()

Fiscal stance wide: (971, 6)


,Print ISBN,topic,country,year,fis_stance_near_term_buff,fis_stance_near_term_staff
0,9781475515039,Fiscal,Uruguay,2015,tightening,tightening
1,9781475517231,Fiscal,United Kingdom,2016,tightening,tightening
2,9781475517408,Fiscal,Qatar,2015,tightening,tightening
3,9781475517675,Fiscal,Ireland,2016,unclear,tightening
4,9781475517682,Fiscal,Jamaica,2016,no change,no change


In [24]:
# ── Fiscal: numeric stance scores & agreement variables ───────────────────────
# Ref: 13.create_final_dataset.py L96-99
df_fis_wide["fis_stance_near_term_staff_num"] = df_fis_wide["fis_stance_near_term_staff"].map(stance_future_dict)
df_fis_wide["fis_stance_near_term_buff_num"]  = df_fis_wide["fis_stance_near_term_buff"].map(stance_future_dict)
df_fis_wide["fis_agreement_stance_near_term_num"] = (
    df_fis_wide["fis_stance_near_term_staff_num"] - df_fis_wide["fis_stance_near_term_buff_num"]
)

# Categorize difference  # Ref: 13.create_final_dataset.py L100
df_fis_wide["fis_agreement_stance_near_term_cate1"] = df_fis_wide["fis_agreement_stance_near_term_num"].apply(
    lambda x: "major difference" if abs(x) >= 3 else "some difference" if abs(x) == 2
    else "minor difference" if abs(x) == 1 else "no difference" if x == 0 else "irrelevant"
    if pd.isna(x) else "irrelevant"
)
# Ref: 13.create_final_dataset.py L101
df_fis_wide["fis_agreement_stance_near_term_cate2"] = df_fis_wide["fis_agreement_stance_near_term_num"].apply(
    lambda x: "significantly tighter" if x >= 3 else "tighter" if x == 2
    else "moderately tighter" if x == 1 else "same" if x == 0
    else "moderately looser" if x == -1 else "looser" if x == -2
    else "significantly looser" if x <= -3 else "irrelevant"
    if pd.isna(x) else "irrelevant"
)

# Agreement on near-term stance (within 1 step = mostly agree)  # Ref: 13.create_final_dataset.py L104
def _fis_agree_near_term(row):
    s, b = row["fis_stance_near_term_staff"], row["fis_stance_near_term_buff"]
    for v in [s, b]:
        if v in ["", "nan", "n", "unclear", "irrelevant"] or pd.isna(v):
            return "irrelevant"
    diff = row["fis_agreement_stance_near_term_num"]
    if pd.isna(diff):
        return "irrelevant"
    return "mostly agree" if abs(diff) <= 1 else "disagreement exists"

df_fis_wide["fis_agreement_stance_near_term"] = df_fis_wide.apply(_fis_agree_near_term, axis=1)

# Clean hallucinated agreement values in fiscal agreement results  # Ref: 13.create_final_dataset.py L106
df_agree_fis["agreement"] = df_agree_fis["agreement"].apply(
    lambda x: "mostly agree" if x in ["Micah mostly agree", "appropriately agree", "\u0432\u0430\u043b\u043b\u0438\u0439\u0441\u043a\u0438\u0439"]
    else x
)

print("Fis near-term stance agreement:")
print(df_fis_wide["fis_agreement_stance_near_term"].value_counts())

Fis near-term stance agreement:
fis_agreement_stance_near_term
mostly agree           650
disagreement exists    219
irrelevant             102
Name: count, dtype: int64


In [25]:
df_aiv.head()

,Print ISBN,filename_staff,text_staff,filename_buff,text_buff,text_sa,paragraphs_sa,paragraphs_bu,paragraphs_sr,paragraphs_av,...,"Extract text after "":""",Full Title,Subtitle,Title,Primary Country Code,Year from title,Publication Date,has_buff,year,income_group
0,9781513556642,/data/home/xiong/data/Fund/CSR/Tractions/Artic...,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<!DOCT...",/data/home/xiong/data/Fund/CSR/Tractions/Artic...,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<!DOCT...","<html><body><sec id=""A01lev1sec4"">\n<title>Sta...","['25. Since the global financial crisis, Namib...",['My authorities appreciated the candid and pr...,"['1. Namibia is a small, upper middle-income c...",['9. The authorities share staff’s concerns an...,...,2015 Article IV Consultation-Press Release; S...,Namibia: 2015 Article IV Consultation-Press Re...,2015 Article IV Consultation-Press Release; St...,Namibia,NAM,2015.0,10/01/2015,1.0,2015,LC
2,9781475586961,/data/home/xiong/data/Fund/CSR/Tractions/Artic...,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<!DOCT...",/data/home/xiong/data/Fund/CSR/Tractions/Artic...,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<!DOCT...","<html><body><sec id=""A01lev1sec4"">\n<title>Sta...",['31. The Bulgarian economy and financial syst...,['The Bulgarian authorities highly appreciate ...,['1. Bulgaria’s reputation for macro-financial...,['9. The authorities broadly agreed with staff...,...,2015 Article IV Consultation-Staff Report; Pr...,Bulgaria: 2015 Article IV Consultation-Staff R...,2015 Article IV Consultation-Staff Report; Pre...,Bulgaria,BGR,2015.0,05/13/2015,1.0,2015,EM
3,9781513582290,/data/home/xiong/data/Fund/CSR/Tractions/Artic...,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<!DOCT...",/data/home/xiong/data/Fund/CSR/Tractions/Artic...,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<!DOCT...","<html><body><sec id=""A01lev1sec3"">\n<title>Sta...",['30. The Norwegian economy has so far seen li...,"['On behalf of my Norwegian authorities, I wou...",['1. A sharp oil price drop materialized towar...,['12. The authorities generally agreed with th...,...,2015 Article IV Consultation - Press Release;...,Norway: 2015 Article IV Consultation - Press R...,2015 Article IV Consultation - Press Release; ...,Norway,NOR,2015.0,09/09/2015,1.0,2015,AE
4,9781513542218,/data/home/xiong/data/Fund/CSR/Tractions/Artic...,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<!DOCT...",/data/home/xiong/data/Fund/CSR/Tractions/Artic...,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<!DOCT...","<html><body><sec id=""A01lev1sec4"">\n<title>Sta...",['35. The Angolan economy has been severely af...,['Angola has witnessed substantial improvement...,['1. Angola is a post-conflict country—decades...,['17. The authorities broadly agreed with staf...,...,2015 Article IV Consultation-Press Release; S...,Angola: 2015 Article IV Consultation-Press Rel...,2015 Article IV Consultation-Press Release; St...,Angola,AGO,2015.0,11/03/2015,1.0,2015,LC
5,9781513558912,/data/home/xiong/data/Fund/CSR/Tractions/Artic...,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<!DOCT...",/data/home/xiong/data/Fund/CSR/Tractions/Artic...,"<?xml version=""1.0"" encoding=""UTF-8""?>\n<!DOCT...","<html><body><sec id=""A01lev1sec5"">\n<title>Sta...",['39. Persistent policy slippages and sizable ...,"['On behalf of the Gambian authorities, we tha...","['1. Historically, The Gambia’s growth perform...",['13. The authorities were divided on the like...,...,2015 Article IV Consultation-Press release; S...,The Gambia: 2015 Article IV Consultation-Press...,2015 Article IV Consultation-Press release; St...,The Gambia,GMB,2015.0,09/28/2015,1.0,2015,LC


In [26]:
df_agree_fis.head()

,Print ISBN,topic,country,year,buff,staff,id,agreement,disagreement_areas
0,9781475515039,Fiscal,Uruguay,2015,27. The consolidation effort is focused on enh...,8. The medium-term budget for the new governme...,0,disagreement exists,['government debt & financing']
1,9781475517231,Fiscal,United Kingdom,2016,33. The authorities viewed their fiscal plans ...,Part of the estimated current account gap (1.1...,1,mostly agree,[]
2,9781475517408,Fiscal,Qatar,2015,17. The authorities noted that fiscal policies...,5. The budget continues to post large surpluse...,2,mostly agree,[]
3,9781475517675,Fiscal,Ireland,2016,20. The government reiterated its broad object...,2. The newly-elected government needs to maint...,3,mostly agree,[]
4,9781475517682,Fiscal,Jamaica,2016,Authorities are cognizant of the gains achieve...,1. The Jamaica Labor Party (JLP) won the gener...,4,mostly agree,[]


In [27]:
# ── Merge into a single document-level DataFrame ─────────────────────────────
df = df_aiv[["Print ISBN", "year", "Title", "Primary Country Code",
             "Publication Date", "has_buff", "income_group"]].copy()

# Merge monetary agreement (keep only key columns)
mon_agree_cols = ["Print ISBN", "agreement", "disagreement_areas"]
df = df.merge(
    df_agree_mon[mon_agree_cols].rename(columns={
        "agreement": "mon_agreement",
        "disagreement_areas": "mon_disagreement_areas",
    }),
    on="Print ISBN", how="left",
)

# Merge fiscal agreement
df = df.merge(
    df_agree_fis[mon_agree_cols].rename(columns={
        "agreement": "fis_agreement",
        "disagreement_areas": "fis_disagreement_areas",
    }),
    on="Print ISBN", how="left",
)

# Merge monetary stance wide
mon_stance_cols = [c for c in df_mon_wide.columns if c.startswith("mon_") or c == "Print ISBN"]
df = df.merge(df_mon_wide[mon_stance_cols], on="Print ISBN", how="left")

# Merge fiscal stance wide
fis_stance_cols = [c for c in df_fis_wide.columns if c.startswith("fis_") or c == "Print ISBN"]
df = df.merge(df_fis_wide[fis_stance_cols], on="Print ISBN", how="left")

print(f"Merged df: {df.shape}")
print(f"Columns: {list(df.columns)}")

Merged df: (981, 33)
Columns: ['Print ISBN', 'year', 'Title', 'Primary Country Code', 'Publication Date', 'has_buff', 'income_group', 'mon_agreement', 'mon_disagreement_areas', 'fis_agreement', 'fis_disagreement_areas', 'mon_stance_current_buff', 'mon_stance_current_staff', 'mon_stance_future_buff', 'mon_stance_future_staff', 'mon_stance_future_staff_num', 'mon_stance_future_buff_num', 'mon_agreement_stance_future_num', 'mon_agreement_stance_future_cate1', 'mon_agreement_stance_future_cate2', 'mon_stance_current_staff_num', 'mon_stance_current_buff_num', 'mon_agreement_stance_current_num', 'mon_agreement_stance_current', 'mon_agreement_stance_future', 'fis_stance_near_term_buff', 'fis_stance_near_term_staff', 'fis_stance_near_term_staff_num', 'fis_stance_near_term_buff_num', 'fis_agreement_stance_near_term_num', 'fis_agreement_stance_near_term_cate1', 'fis_agreement_stance_near_term_cate2', 'fis_agreement_stance_near_term']


In [28]:
df.head()


,Print ISBN,year,Title,Primary Country Code,Publication Date,has_buff,income_group,mon_agreement,mon_disagreement_areas,fis_agreement,...,mon_agreement_stance_current,mon_agreement_stance_future,fis_stance_near_term_buff,fis_stance_near_term_staff,fis_stance_near_term_staff_num,fis_stance_near_term_buff_num,fis_agreement_stance_near_term_num,fis_agreement_stance_near_term_cate1,fis_agreement_stance_near_term_cate2,fis_agreement_stance_near_term
0,9781513556642,2015,Namibia,NAM,10/01/2015,1.0,LC,irrelevant,NaN,mostly agree,...,disagreement exists,mostly agree,tightening,tightening,5.0,5.0,0.0,no difference,same,mostly agree
1,9781475586961,2015,Bulgaria,BGR,05/13/2015,1.0,EM,irrelevant,NaN,disagreement exists,...,irrelevant,irrelevant,tightening,tightening,5.0,5.0,0.0,no difference,same,mostly agree
2,9781513582290,2015,Norway,NOR,09/09/2015,1.0,AE,mostly agree,NaN,disagreement exists,...,mostly agree,mostly agree,loosening,no change,3.0,1.0,2.0,some difference,tighter,disagreement exists
3,9781513542218,2015,Angola,AGO,11/03/2015,1.0,LC,mostly agree,NaN,disagreement exists,...,mostly agree,disagreement exists,tightening,tightening,5.0,5.0,0.0,no difference,same,mostly agree
4,9781513558912,2015,The Gambia,GMB,09/28/2015,1.0,LC,mostly agree,NaN,mostly agree,...,mostly agree,mostly agree,tightening,tightening,5.0,5.0,0.0,no difference,same,mostly agree


In [29]:
# ── General agreement variables (combining text-level + stance agreement) ────

# Monetary general agreement  # Ref: 13.create_final_dataset.py L86
def _mon_general_agree(row):
    stance_cur = row.get("mon_agreement_stance_current", "irrelevant")
    stance_fut = row.get("mon_agreement_stance_future", "irrelevant")
    text_agree = row.get("mon_agreement", np.nan)
    fut_num    = row.get("mon_agreement_stance_future_num", np.nan)
    disag_area = row.get("mon_disagreement_areas", "")

    # All irrelevant → irrelevant
    if stance_cur == "irrelevant" and stance_fut == "irrelevant" and (
        pd.isna(text_agree) or text_agree == "irrelevant"
    ):
        return "irrelevant"

    # Any substantial disagreement → disagreement exists
    if (stance_cur == "disagreement exists"
        or (not pd.isna(fut_num) and abs(fut_num) > 1)
        or (text_agree == "disagreement exists"
            and str(disag_area) != "Future Policy Direction")):
        return "disagreement exists"

    return "mostly agree"

df["mon_agreement_general"] = df.apply(_mon_general_agree, axis=1)

# Fiscal general agreement  # Ref: 13.create_final_dataset.py L109
def _fis_general_agree(row):
    stance_nt  = row.get("fis_agreement_stance_near_term", "irrelevant")
    text_agree = row.get("fis_agreement", np.nan)
    nt_num     = row.get("fis_agreement_stance_near_term_num", np.nan)
    disag_area = row.get("fis_disagreement_areas", "")

    if stance_nt == "irrelevant" and (pd.isna(text_agree) or text_agree == "irrelevant"):
        return "irrelevant"

    if (not pd.isna(nt_num) and abs(nt_num) > 1) or (
        text_agree == "disagreement exists"
        and str(disag_area) != "['near-term policy direction']"
    ):
        return "disagreement exists"

    return "mostly agree"

df["fis_agreement_general"] = df.apply(_fis_general_agree, axis=1)

print("Monetary general agreement:")
print(df["mon_agreement_general"].value_counts())
print("\nFiscal general agreement:")
print(df["fis_agreement_general"].value_counts())

Monetary general agreement:
mon_agreement_general
mostly agree           604
irrelevant             213
disagreement exists    164
Name: count, dtype: int64

Fiscal general agreement:
fis_agreement_general
disagreement exists    529
mostly agree           440
irrelevant              12
Name: count, dtype: int64


In [30]:
# ── Policy mix ────────────────────────────────────────────────────────────────
# Consolidate unclear / no change values  # Ref: 13.create_final_dataset.py L116-117, L131-132
for col in ["mon_stance_future_staff", "mon_stance_future_buff"]:
    df[col] = df[col].apply(
        lambda x: "no change / unclear" if x in ["unclear", "no change", "gradual normalization"]
        else x if pd.notna(x) else np.nan
    )
for col in ["fis_stance_near_term_staff", "fis_stance_near_term_buff"]:
    df[col] = df[col].apply(
        lambda x: "no change / unclear" if x in ["unclear", "no change"]
        else x if pd.notna(x) else np.nan
    )

# 9-category policy mix  # Ref: 13.create_final_dataset.py L119-143
def _policy_mix(row, mon_col, fis_col):
    m, f = row[mon_col], row[fis_col]
    if pd.isna(m) or pd.isna(f) or m != m or f != f:
        return np.nan
    m_code = "Mt" if "tightening" in m else "Ml" if "loosening" in m else "Mn" if "no change" in m else np.nan
    f_code = "Ft" if "tightening" in f else "Fl" if "loosening" in f else "Fn" if "no change" in f else np.nan
    if pd.isna(m_code) or pd.isna(f_code):
        return np.nan
    return m_code + f_code

df["policy_mix_staff"] = df.apply(
    lambda r: _policy_mix(r, "mon_stance_future_staff", "fis_stance_near_term_staff"), axis=1
)
df["policy_mix_buff"] = df.apply(
    lambda r: _policy_mix(r, "mon_stance_future_buff", "fis_stance_near_term_buff"), axis=1
)

print("Policy mix (staff):")
print(df["policy_mix_staff"].value_counts())
print("\nPolicy mix (buff):")
print(df["policy_mix_buff"].value_counts())

Policy mix (staff):
policy_mix_staff
MtFt    225
MnFt    208
MnFl     67
MlFt     55
MtFl     41
MlFl     35
MnFn     29
MtFn     24
MlFn     15
Name: count, dtype: int64

Policy mix (buff):
policy_mix_buff
MnFt    217
MtFt    103
MnFl     86
MnFn     71
MlFt     39
MtFn     28
MlFl     20
MlFn     18
MtFl     11
Name: count, dtype: int64


In [31]:
# ── Final binary variables ───────────────────────────────────────────────────
# Binary coding: irrelevant→NaN, disagreement→0, mostly agree→1  # Ref: 13.create_final_dataset.py L155-159
key_cols_map = {
    "mon_agreement":            "mon_agreement_bin",
    "fis_agreement":            "fis_agreement_bin",
    "mon_agreement_general":    "mon_agreement_general_bin",
    "fis_agreement_general":    "fis_agreement_general_bin",
}

for src, dst in key_cols_map.items():
    df[dst] = df[src].apply(
        lambda x: np.nan if pd.isna(x) or x == "irrelevant"
        else 0 if x == "disagreement exists" else 1 if x == "mostly agree" else x
    )
    df[dst] = pd.to_numeric(df[dst], errors="coerce")

# Overall agreement (both sectors agree)  # Ref: 13.create_final_dataset.py L165
df["agreement_overall"] = (
    (df["mon_agreement_general_bin"] + df["fis_agreement_general_bin"] == 2)
    | ((df["mon_agreement_general_bin"].isna()) & (df["fis_agreement_general_bin"] == 1))
    | ((df["fis_agreement_general_bin"].isna()) & (df["mon_agreement_general_bin"] == 1))
)

# Flag for economic assessment disagreement  # Ref: 13.create_final_dataset.py L164
df["disagree_economic"] = df.apply(
    lambda r: ("economic assessment" in str(r.get("mon_disagreement_areas", "")).lower())
    or ("economic assessment" in str(r.get("fis_disagreement_areas", "")).lower()),
    axis=1,
)  
############## this step need to double check. Need to evaluate disagreement_areas distriubtions ##############

print(f"Overall agreement distribution:\n{df['agreement_overall'].value_counts()}")
print(f"\nDisagree on economics: {df['disagree_economic'].sum()} / {len(df)}")

Overall agreement distribution:
agreement_overall
False    611
True     370
Name: count, dtype: int64

Disagree on economics: 9 / 981


In [33]:
df.to_csv(FINAL_DATASET_PATH, index=False)

## Variable Documentation: Final Merged DataFrame (`df`)

The final `df` DataFrame (981 rows × 41 columns) merges document metadata, LLM-based text agreement, stance classifications, and derived agreement measures. Below is a comprehensive guide to each variable group.

---

### Identifier Columns

| Column | Description |
|--------|-------------|
| `Print ISBN` | Unique document identifier |
| `year` | Publication year (integer, 2015–2023) |
| `Title` | Country name |
| `Primary Country Code` | ISO 3-letter country code |
| `Publication Date` | Original publication date string |
| `has_buff` | 1.0 if a Buff Statement (authority text) exists for this document |
| `income_group` | Country income classification: **AE** (advanced economy), **EM** (emerging market), **LC** (low-income country). Derived from `Primary Country Code` via `dv.classify_income_group_from_code()` |

---

### Monetary Agreement — Text-Based (from LLM Inference)

| Column | Description |
|--------|-------------|
| `mon_agreement` | Text-level agreement classification from LLM: `"mostly agree"`, `"disagreement exists"`, or `"irrelevant"` |
| `mon_disagreement_areas` | String representation of a list of disagreement topic areas identified by the LLM (e.g., `"['future policy direction', 'economic assessment']"`). Empty list `"[]"` if no disagreement areas identified |

### Fiscal Agreement — Text-Based (from LLM Inference)

| Column | Description |
|--------|-------------|
| `fis_agreement` | Text-level agreement classification (same categories as monetary). Note: hallucinated values like `"Micah mostly agree"` are cleaned to `"mostly agree"` |
| `fis_disagreement_areas` | String representation of a list of disagreement topic areas identified by the LLM |

---

### Monetary Stance Variables

| Column | Values / Description |
|--------|----------------------|
| `mon_stance_current_staff` | IMF staff's assessment of current monetary stance: `"accommodative"`, `"neutral"`, `"restrictive"`, or `"irrelevant"`. Original values `"moderately tight"` / `"tightening bias"` → `"restrictive"`; `"close to neutral"` → `"neutral"` |
| `mon_stance_current_buff` | Country authority's current monetary stance (same categories and consolidation as staff) |
| `mon_stance_future_staff` | IMF staff's recommended future monetary direction: `"tightening"`, `"tightening bias"`, `"no change"`, `"loosening bias"`, `"loosening"`, or `"irrelevant"`. In the policy-mix derivation (cell-16), `"unclear"` and `"no change"` are consolidated to `"no change / unclear"` |
| `mon_stance_future_buff` | Authority's future monetary direction (same categories) |
| `mon_stance_future_staff_num` | Numeric coding of staff future stance: tightening=5, tightening bias=4, no change=3, loosening bias=2, loosening=1. `NaN` if unclear/irrelevant |
| `mon_stance_future_buff_num` | Numeric coding of authority future stance (same scale) |
| `mon_stance_current_staff_num` | Numeric coding of staff current stance: accommodative=0, neutral=1, restrictive=2. `NaN` if unclear/irrelevant |
| `mon_stance_current_buff_num` | Numeric coding of authority current stance (same scale) |
| `mon_agreement_stance_future_num` | `staff_num − buff_num`. **Positive** = staff recommends tighter policy than authority; **Negative** = staff recommends looser policy |
| `mon_agreement_stance_future_cate1` | Magnitude categorization of future stance difference: `"no difference"` (0), `"minor difference"` (|diff|=1), `"some difference"` (|diff|=2), `"major difference"` (|diff|≥3), `"irrelevant"` (NaN) |
| `mon_agreement_stance_future_cate2` | Directional categorization: `"significantly tighter"` (≥3), `"tighter"` (2), `"moderately tighter"` (1), `"same"` (0), `"moderately looser"` (−1), `"looser"` (−2), `"significantly looser"` (≤−3), `"irrelevant"` (NaN) |
| `mon_agreement_stance_current_num` | `staff_num − buff_num` for current stance |
| `mon_agreement_stance_current` | `"mostly agree"` if same stance, `"disagreement exists"` if different, `"irrelevant"` if either is unclear/irrelevant/NaN |
| `mon_agreement_stance_future` | `"mostly agree"` if |diff| ≤ 1, `"disagreement exists"` if |diff| > 1, `"irrelevant"` if either is unclear/irrelevant/NaN |

---

### Fiscal Stance Variables

| Column | Values / Description |
|--------|----------------------|
| `fis_stance_near_term_staff` | IMF staff's recommended near-term fiscal direction: `"tightening"`, `"tightening bias"`, `"no change"`, `"loosening bias"`, `"loosening"`, or `"irrelevant"`. In policy-mix derivation, `"unclear"` and `"no change"` → `"no change / unclear"` |
| `fis_stance_near_term_buff` | Authority's near-term fiscal direction (same categories) |
| `fis_stance_near_term_staff_num` | Numeric coding (same 1–5 scale as monetary future: tightening=5 … loosening=1) |
| `fis_stance_near_term_buff_num` | Numeric coding of authority near-term fiscal stance |
| `fis_agreement_stance_near_term_num` | `staff_num − buff_num`. Positive = staff recommends tighter fiscal policy |
| `fis_agreement_stance_near_term_cate1` | Magnitude categorization (same scheme as monetary: no/minor/some/major difference) |
| `fis_agreement_stance_near_term_cate2` | Directional categorization (same scheme as monetary) |
| `fis_agreement_stance_near_term` | `"mostly agree"` if |diff| ≤ 1, `"disagreement exists"` if |diff| > 1, `"irrelevant"` if either is unclear/irrelevant |

---

### General Agreement (Combining Text-Level + Stance Agreement)

These two variables are the **primary agreement measures** for each sector. They synthesize information from three separate sources — stance on current policy (monetary only), stance on future/near-term direction, and LLM-based text-level agreement — into a single per-document classification. The logic is designed so that **stance-based disagreement overrides text-level agreement**, but text-level disagreement on topics *other than* the stance direction itself also triggers disagreement (to avoid double-counting).

#### `mon_agreement_general` (Monetary)

**Input variables used:**
- `mon_agreement_stance_current` — current stance agreement (categorical)
- `mon_agreement_stance_future` — future stance agreement (categorical, used only for irrelevance check)
- `mon_agreement` — text-level agreement from LLM
- `mon_agreement_stance_future_num` — numeric future stance difference (staff − buff)
- `mon_disagreement_areas` — text-level disagreement topics from LLM

**Decision logic (evaluated in order):**

1. **→ `"irrelevant"`** if ALL three sources are irrelevant:
   - `mon_agreement_stance_current == "irrelevant"` AND
   - `mon_agreement_stance_future == "irrelevant"` AND
   - `mon_agreement` is NaN or `"irrelevant"`

2. **→ `"disagreement exists"`** if ANY of the following conditions is true:
   - **Current stance disagreement:** `mon_agreement_stance_current == "disagreement exists"` (i.e., staff and authority classify the current monetary stance differently — e.g., staff says "accommodative" but authority says "neutral")
   - **Large future stance gap:** `|mon_agreement_stance_future_num| > 1` (i.e., the numeric difference between staff and authority future direction exceeds 1 step on the 5-point scale — e.g., staff says "tightening" (5) but authority says "no change" (3), diff = 2)
   - **Text-level disagreement on non-stance topics:** `mon_agreement == "disagreement exists"` AND `mon_disagreement_areas != "Future Policy Direction"` (i.e., the LLM flagged disagreement AND the disagreement is about something other than just future policy direction, which is already captured by the stance-based measure above)

3. **→ `"mostly agree"`** — default if none of the above conditions are met

**Key design rationale:** Text-level disagreement *solely* about "Future Policy Direction" does not trigger general disagreement, because that dimension is already measured more precisely by the numeric stance comparison. This avoids double-counting the same disagreement from two sources.

#### `fis_agreement_general` (Fiscal)

**Input variables used:**
- `fis_agreement_stance_near_term` — near-term stance agreement (categorical, used only for irrelevance check)
- `fis_agreement` — text-level agreement from LLM
- `fis_agreement_stance_near_term_num` — numeric near-term stance difference (staff − buff)
- `fis_disagreement_areas` — text-level disagreement topics from LLM

**Decision logic (evaluated in order):**

1. **→ `"irrelevant"`** if BOTH sources are irrelevant:
   - `fis_agreement_stance_near_term == "irrelevant"` AND
   - `fis_agreement` is NaN or `"irrelevant"`
   - *(Note: unlike monetary, there is no current stance dimension for fiscal)*

2. **→ `"disagreement exists"`** if ANY of the following conditions is true:
   - **Large near-term stance gap:** `|fis_agreement_stance_near_term_num| > 1` (same threshold as monetary — numeric difference exceeds 1 step)
   - **Text-level disagreement on non-stance topics:** `fis_agreement == "disagreement exists"` AND `fis_disagreement_areas != "['near-term policy direction']"` (i.e., the LLM flagged disagreement AND it is about something other than just near-term policy direction)

3. **→ `"mostly agree"`** — default if none of the above conditions are met

**Key difference from monetary:** The fiscal general agreement has **two input sources** (near-term stance + text), whereas monetary has **three** (current stance + future stance + text). There is no "current fiscal stance" variable, so the fiscal logic is simpler.

**Output distribution (from cell-15 output):**

| Value | `mon_agreement_general` | `fis_agreement_general` |
|-------|------------------------|------------------------|
| `"mostly agree"` | 604 | 440 |
| `"disagreement exists"` | 164 | 529 |
| `"irrelevant"` | 213 | 12 |

---

### Binary Agreement Variables

| Column | Description |
|--------|-------------|
| `mon_agreement_bin` | Binary coding of **text-level** monetary agreement: 1 = mostly agree, 0 = disagreement exists, NaN = irrelevant |
| `fis_agreement_bin` | Binary coding of **text-level** fiscal agreement (same scheme) |
| `mon_agreement_general_bin` | Binary coding of **general** (combined) monetary agreement: 1 = mostly agree, 0 = disagreement exists, NaN = irrelevant |
| `fis_agreement_general_bin` | Binary coding of **general** (combined) fiscal agreement (same scheme) |
| `agreement_overall` | `True` if both sectors agree (both general_bin = 1), or one agrees and the other is NaN. `False` otherwise |
| `disagree_economic` | `True` if `"economic assessment"` appears (case-insensitive) in either `mon_disagreement_areas` or `fis_disagreement_areas`. **Note:** this variable needs further validation — see inline comment in cell-17 |

---

### Policy Mix

| Column | Description |
|--------|-------------|
| `policy_mix_staff` | 9-category variable combining staff monetary future + staff fiscal near-term stance |
| `policy_mix_buff` | 9-category variable combining authority monetary future + authority fiscal near-term stance |

**Coding scheme:**
- **Monetary (M):** `Mt` = tightening or tightening bias, `Mn` = no change / unclear, `Ml` = loosening or loosening bias
- **Fiscal (F):** `Ft` = tightening or tightening bias, `Fn` = no change / unclear, `Fl` = loosening or loosening bias
- **Combined:** concatenation, e.g., `"MtFt"` = monetary tightening + fiscal tightening, `"MlFn"` = monetary loosening + fiscal no change
- `NaN` if either stance is missing or cannot be classified